# common/data_io — Đọc dữ liệu, schema cột, mã băm và ghi JSON

Notebook chứa hàm dùng chung, không tự train và không tự ghi kết quả. Các notebook chính nạp tự động bằng `%run`. Khi sửa hàm, khởi động lại kernel rồi Run All notebook chính.


## Thư viện và hằng số
Tên cột trong `CATEGORICAL` đã strip khoảng trắng/tab, vì `load_data` strip tên cột trước khi dùng.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

LABELS = ['Dropout', 'Enrolled', 'Graduate']
TARGET = 'Target'

CATEGORICAL = ['Marital status', 'Application mode', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]

BINARY = ['Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']


## `validate_columns`
Báo lỗi nếu cột khai báo trong `CATEGORICAL`/`BINARY` không có trong CSV, tránh việc cột sai tên lặng lẽ rơi vào nhóm số.

In [ ]:
def validate_columns(df):
    missing = [c for c in CATEGORICAL + BINARY if c not in df.columns]
    if missing:
        raise ValueError(f'Cột khai báo nhưng không có trong CSV: {missing}')


## `load_data`
Đọc CSV phân tách bằng dấu chấm phẩy, kiểm tra nhãn và danh sách cột.

In [ ]:
def load_data(path):
    path = Path(path)
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    if TARGET not in df or len(df.columns) < 2:
        raise ValueError('Expected UCI CSV separated by semicolons, with a Target column.')
    if df[TARGET].isna().any() or not set(df[TARGET].unique()).issubset(LABELS):
        raise ValueError('Target must contain only Dropout, Enrolled and Graduate, without missing values.')
    validate_columns(df)
    return df


## `fingerprint`
Tính mã băm để phát hiện CSV bị thay đổi.

In [ ]:
def fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


## `save_json`
Lưu cấu hình hoặc kết quả dạng JSON.

In [ ]:
def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=lambda x: float(x)), encoding='utf-8')
